## Importaciones y configuración inicial

In [ ]:
# Importaciones generales
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import json
import random
from datetime import datetime

# Utilidades para trabajar con audio en memoria
import io

# Dataset de Hugging Face
from datasets import load_from_disk, DatasetDict, Audio

# Visualización
import matplotlib.pyplot as plt

# Audio
from IPython.display import Audio as IPythonAudio, display
import librosa

# PyTorch
import torch

# Whisper / Transformers
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Métricas
import evaluate

# Utilidades para entrenamiento
from dataclasses import dataclass
from typing import Any, Dict, List, Union

In [ ]:
# Configuración general
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Rutas principales del proyecto
RUTA_DATASET_ORIGINAL = Path(r"C:\Lara\datasets\lara_whisper_dataset")
RUTA_DATASET_PREPARADO = Path(r"C:\Lara\datasets\lara_whisper_dataset_preparado")

RUTA_RESULTADOS_CUADERNO_03 = Path(r"C:\Lara\resultados\03_mejora_modelo")
RUTA_RESULTADOS_CUADERNO_04 = Path(r"C:\Lara\resultados\04_analisis_errores_modelo_mejorado")
RUTA_RESULTADOS_CUADERNO_05 = Path(r"C:\Lara\resultados\05_limpieza_dataset")

RUTA_DATASET_LIMPIO_V2 = Path(r"C:\Lara\datasets\lara_whisper_dataset_limpio_v2")
RUTA_DATASET_LIMPIO_V2_SPLITS = Path(r"C:\Lara\datasets\lara_whisper_dataset_limpio_v2_split")

RUTA_MODELO_MEJORADO_ANTERIOR = Path(r"C:\Lara\modelos_entrenados\whisper_base_mejorado")
RUTA_MODELO_REENTRENADO_V2 = Path(r"C:\Lara\modelos_entrenados\whisper_base_limpio_v2")
RUTA_MODELO_BASE = "openai/whisper-base"

RUTA_RESULTADOS_CUADERNO_06 = Path(r"C:\Lara\resultados\06_reentrenamiento_limpio_v2")

RUTA_BASE_AUDIOS = Path(r"C:\Lara\audios-230426\audios-s3")

# Creamos carpetas de salida del cuaderno 06
RUTA_RESULTADOS_CUADERNO_06.mkdir(parents=True, exist_ok=True)
RUTA_MODELO_REENTRENADO_V2.mkdir(parents=True, exist_ok=True)

# Parámetros de entrenamiento
NUM_TRAIN_EPOCHS = 8
LEARNING_RATE = 2e-6

TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2

WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

GENERATION_MAX_LENGTH = 225
LOGGING_STEPS = 100
SAVE_TOTAL_LIMIT = 2


print("Dispositivo:", device)
print("Ruta dataset original:", RUTA_DATASET_ORIGINAL)
print("Ruta dataset preparado:", RUTA_DATASET_PREPARADO)
print("Ruta resultados cuaderno 03:", RUTA_RESULTADOS_CUADERNO_03)
print("Ruta resultados cuaderno 04:", RUTA_RESULTADOS_CUADERNO_04)
print("Ruta resultados cuaderno 05:", RUTA_RESULTADOS_CUADERNO_05)
print("Ruta dataset limpio v2:", RUTA_DATASET_LIMPIO_V2)
print("Ruta dataset limpio v2 splits:", RUTA_DATASET_LIMPIO_V2_SPLITS)
print("Ruta modelo mejorado anterior:", RUTA_MODELO_MEJORADO_ANTERIOR)
print("Ruta modelo reentrenado v2:", RUTA_MODELO_REENTRENADO_V2)
print("Ruta resultados cuaderno 06:", RUTA_RESULTADOS_CUADERNO_06)
print("Ruta base audios:", RUTA_BASE_AUDIOS)

Dispositivo: cuda
Ruta dataset original: C:\Lara\datasets\lara_whisper_dataset
Ruta dataset preparado: C:\Lara\datasets\lara_whisper_dataset_preparado
Ruta resultados cuaderno 03: C:\Lara\resultados\03_mejora_modelo
Ruta resultados cuaderno 04: C:\Lara\resultados\04_analisis_errores_modelo_mejorado
Ruta resultados cuaderno 05: C:\Lara\resultados\05_limpieza_dataset
Ruta dataset limpio v2: C:\Lara\datasets\lara_whisper_dataset_limpio_v2
Ruta dataset limpio v2 splits: C:\Lara\datasets\lara_whisper_dataset_limpio_v2_split
Ruta modelo mejorado anterior: C:\Lara\modelos_entrenados\whisper_base_mejorado
Ruta modelo reentrenado v2: C:\Lara\modelos_entrenados\whisper_base_limpio_v2
Ruta resultados cuaderno 06: C:\Lara\resultados\06_reentrenamiento_limpio_v2
Ruta base audios: C:\Lara\audios-230426\audios-s3


## Carga del dataset limpio v2 con splits

In [7]:
# Cargamos el dataset limpio v2 con los splits originales filtrados
dataset_v2_splits = load_from_disk(str(RUTA_DATASET_LIMPIO_V2_SPLITS))

dataset_v2_splits

DatasetDict({
    train: Dataset({
        features: ['audio', 'texto', 'indice_original'],
        num_rows: 32673
    })
    test: Dataset({
        features: ['audio', 'texto', 'indice_original'],
        num_rows: 6512
    })
    eval: Dataset({
        features: ['audio', 'texto', 'indice_original'],
        num_rows: 1634
    })
})

In [8]:
# Revisamos los splits disponibles y el número de registros
for split in dataset_v2_splits.keys():
    print("Split:", split)
    print("Registros:", len(dataset_v2_splits[split]))
    print("Columnas:", dataset_v2_splits[split].column_names)
    print("----------------------------------------")

Split: train
Registros: 32673
Columnas: ['audio', 'texto', 'indice_original']
----------------------------------------
Split: test
Registros: 6512
Columnas: ['audio', 'texto', 'indice_original']
----------------------------------------
Split: eval
Registros: 1634
Columnas: ['audio', 'texto', 'indice_original']
----------------------------------------


In [10]:
# Mostramos una primera muestra de cada split sin imprimir el audio completo
for split in dataset_v2_splits.keys():
    muestra = dataset_v2_splits[split][0]
    audio = muestra["audio"]

    print("Split:", split)
    print("Texto:", muestra["texto"])
    print("Índice original:", muestra["indice_original"])
    print("Claves audio:", audio.keys())

    if audio.get("path") is not None:
        print("Ruta audio:", audio["path"])

    if audio.get("bytes") is not None:
        print("Audio en bytes:", len(audio["bytes"]), "bytes")

    print("----------------------------------------")

Split: train
Texto: EL MAGO CONSIGUIÓ QUE EL ÁGUILA LLEGARÁ A LA LAGUNA.
Índice original: 8776
Claves audio: dict_keys(['bytes', 'path'])
Ruta audio: 6564622239048d0f405c669c_1716799914.wav
Audio en bytes: 213925 bytes
----------------------------------------
Split: test
Texto: EN LA LATA HAY LIMONADA.
Índice original: 29890
Claves audio: dict_keys(['bytes', 'path'])
Ruta audio: 662819171ef22d020bf25236_1744267180.wav
Audio en bytes: 28354 bytes
----------------------------------------
Split: eval
Texto: YO EMPUJO UN PAQUETE.
Índice original: 9714
Claves audio: dict_keys(['bytes', 'path'])
Ruta audio: 6564622239048d0f405c669c_1717406129.wav
Audio en bytes: 73313 bytes
----------------------------------------


In [11]:
# Cargamos el audio a partir del nombre guardado en el dataset
def cargar_audio_dataset(audio):
    ruta_audio = RUTA_BASE_AUDIOS / audio["path"]

    audio_array, sampling_rate = librosa.load(
        ruta_audio,
        sr=16000
    )

    return audio_array, sampling_rate

In [12]:
# Probamos la carga de audio con una muestra del split train
muestra_prueba = dataset_v2_splits["train"][0]

audio_array, sampling_rate = cargar_audio_dataset(muestra_prueba["audio"])

print("Sampling rate:", sampling_rate)
print("Forma audio:", audio_array.shape)
print("Duración aproximada:", round(len(audio_array) / sampling_rate, 2), "segundos")

display(IPythonAudio(audio_array, rate=sampling_rate))

C:\Users\abelg\AppData\Local\Temp\ipykernel_20536\1910533709.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(
c:\Users\abelg\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Sampling rate: 16000
Forma audio: (211200,)
Duración aproximada: 13.2 segundos


## Carga del procesador Whisper

In [15]:
# Cargamos el procesador de Whisper para transcripción en español
processor = WhisperProcessor.from_pretrained(
    RUTA_MODELO_BASE,
    language="Spanish",
    task="transcribe"
)

feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

In [18]:
# Cargamos el modelo mejorado anterior como punto de partida
model = WhisperForConditionalGeneration.from_pretrained(
    str(RUTA_MODELO_MEJORADO_ANTERIOR)
)

model = model.to(device)

print("Modelo cargado correctamente")
print("Dispositivo:", device)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Modelo cargado correctamente
Dispositivo: cuda


In [19]:
# Configuramos la generación para transcripción en español
model.generation_config.language = "spanish"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="spanish",
    task="transcribe"
)

model.generation_config.suppress_tokens = []

# Desactivamos caché durante entrenamiento
model.config.use_cache = False

## Preparación / preprocesado del dataset

In [20]:
# Preparamos cada muestra para Whisper
def preparar_dataset(batch):
    audio_array, sampling_rate = cargar_audio_dataset(batch["audio"])

    batch["input_features"] = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate
    ).input_features[0]

    batch["labels"] = tokenizer(batch["texto"]).input_ids

    return batch

In [21]:
# Probamos el preprocesado con una muestra
muestra_preprocesada = preparar_dataset(dataset_v2_splits["train"][0])

print("Claves:", muestra_preprocesada.keys())
print("Forma input_features:", np.array(muestra_preprocesada["input_features"]).shape)
print("Longitud labels:", len(muestra_preprocesada["labels"]))
print("Texto:", dataset_v2_splits["train"][0]["texto"])

C:\Users\abelg\AppData\Local\Temp\ipykernel_20536\1910533709.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(
c:\Users\abelg\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Claves: dict_keys(['audio', 'texto', 'indice_original', 'input_features', 'labels'])
Forma input_features: (80, 3000)
Longitud labels: 31
Texto: EL MAGO CONSIGUIÓ QUE EL ÁGUILA LLEGARÁ A LA LAGUNA.


In [22]:
# Aplicamos el preprocesado a los splits del dataset limpio v2
columnas_originales = dataset_v2_splits["train"].column_names

dataset_v2_preparado = DatasetDict()

for split in dataset_v2_splits.keys():
    print("Preparando split:", split)

    dataset_v2_preparado[split] = dataset_v2_splits[split].map(
        preparar_dataset,
        remove_columns=columnas_originales,
        load_from_cache_file=False
    )

    print("Split preparado:", split)
    print("Registros:", len(dataset_v2_preparado[split]))
    print("----------------------------------------")

Preparando split: train


Map:   0%|          | 0/32673 [00:00<?, ? examples/s]

C:\Users\abelg\AppData\Local\Temp\ipykernel_20536\1910533709.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(


Split preparado: train
Registros: 32673
----------------------------------------
Preparando split: test


Map:   0%|          | 0/6512 [00:00<?, ? examples/s]

Split preparado: test
Registros: 6512
----------------------------------------
Preparando split: eval


Map:   0%|          | 0/1634 [00:00<?, ? examples/s]

Split preparado: eval
Registros: 1634
----------------------------------------


In [23]:
# Revisamos el dataset preparado
dataset_v2_preparado

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 32673
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 6512
    })
    eval: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1634
    })
})

## Configuración del entrenamiento

In [24]:
# Cargamos las métricas de evaluación
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [35]:
# Calculamos WER y CER durante la evaluación
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Sustituimos los valores ignorados para poder decodificar las etiquetas
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

In [26]:
# Creamos el data collator para preparar los lotes de entrenamiento
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        # Eliminamos el token inicial si aparece al principio de todas las etiquetas
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [27]:
# Inicializamos el data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

In [33]:
# Calculamos los pasos aproximados de entrenamiento y calentamiento
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

steps_por_epoch = len(dataset_v2_preparado["train"]) // EFFECTIVE_BATCH_SIZE
total_steps = steps_por_epoch * NUM_TRAIN_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print("Batch efectivo:", EFFECTIVE_BATCH_SIZE)
print("Steps por epoch:", steps_por_epoch)
print("Total steps:", total_steps)
print("Warmup steps:", warmup_steps)

Batch efectivo: 8
Steps por epoch: 4084
Total steps: 32672
Warmup steps: 1633


In [34]:
# Configuramos los argumentos del entrenamiento
training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUTA_RESULTADOS_CUADERNO_06 / "trainer_output"),

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),

    eval_strategy="epoch",
    save_strategy="epoch",

    predict_with_generate=True,
    generation_max_length=GENERATION_MAX_LENGTH,

    logging_steps=LOGGING_STEPS,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    save_total_limit=SAVE_TOTAL_LIMIT,

    report_to="none",

    seed=SEED
)

## Entrenamiento del modelo con dataset limpio v2

In [37]:
# Creamos el trainer para el reentrenamiento
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset_v2_preparado["train"],
    eval_dataset=dataset_v2_preparado["eval"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [38]:
# Entrenamos el modelo con el dataset limpio v2
resultado_train = trainer.train()

resultado_train

Epoch,Training Loss,Validation Loss,Wer,Cer
1,0.001939,0.142275,0.460506,0.337071
2,0.000602,0.145495,0.496520,0.368321
3,0.000679,0.143626,0.462927,0.345002
4,0.000350,0.145580,0.446182,0.326370
5,0.000057,0.145948,0.469989,0.352136
6,0.000048,0.146730,0.451528,0.330734
7,0.000034,0.146825,0.448502,0.329975
8,0.000034,0.147089,0.451125,0.331113


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=32680, training_loss=0.0004889874759907042, metrics={'train_runtime': 39573.7835, 'train_samples_per_second': 6.605, 'train_steps_per_second': 0.826, 'total_flos': 1.695336523628544e+19, 'train_loss': 0.0004889874759907042, 'epoch': 8.0})

In [39]:
# Guardamos el mejor modelo reentrenado con dataset limpio v2
trainer.save_model(str(RUTA_MODELO_REENTRENADO_V2))
processor.save_pretrained(str(RUTA_MODELO_REENTRENADO_V2))

print("Modelo guardado en:", RUTA_MODELO_REENTRENADO_V2)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en: C:\Lara\modelos_entrenados\whisper_base_limpio_v2


In [40]:
# Evaluamos el mejor modelo cargado sobre el split de test limpio v2
metricas_test_v2 = trainer.evaluate(
    eval_dataset=dataset_v2_preparado["test"],
    metric_key_prefix="test"
)

metricas_test_v2

Training Loss,Validation Loss,Epoch,Wer,Cer
0.000034,0.146049,8,0.480188,0.354950


{'test_loss': 0.14604948461055756,
 'test_wer': 0.48018848804215647,
 'test_cer': 0.35495029953511387}

In [41]:
# Guardamos las métricas de evaluación sobre test
ruta_metricas_test_v2 = RUTA_RESULTADOS_CUADERNO_06 / "metricas_test_limpio_v2.json"

with open(ruta_metricas_test_v2, "w", encoding="utf-8") as f:
    json.dump(metricas_test_v2, f, indent=4, ensure_ascii=False)

print("Métricas guardadas en:", ruta_metricas_test_v2)
metricas_test_v2

Métricas guardadas en: C:\Lara\resultados\06_reentrenamiento_limpio_v2\metricas_test_limpio_v2.json


{'test_loss': 0.14604948461055756,
 'test_wer': 0.48018848804215647,
 'test_cer': 0.35495029953511387}

In [42]:
# Guardamos las métricas generales del entrenamiento
metricas_train_v2 = resultado_train.metrics

ruta_metricas_train_v2 = RUTA_RESULTADOS_CUADERNO_06 / "metricas_entrenamiento_limpio_v2.json"

with open(ruta_metricas_train_v2, "w", encoding="utf-8") as f:
    json.dump(metricas_train_v2, f, indent=4, ensure_ascii=False)

print("Métricas guardadas en:", ruta_metricas_train_v2)
metricas_train_v2

Métricas guardadas en: C:\Lara\resultados\06_reentrenamiento_limpio_v2\metricas_entrenamiento_limpio_v2.json


{'train_runtime': 39573.7835,
 'train_samples_per_second': 6.605,
 'train_steps_per_second': 0.826,
 'total_flos': 1.695336523628544e+19,
 'train_loss': 0.0004889874759907042,
 'epoch': 8.0}

In [43]:
# Guardamos el histórico completo del entrenamiento
df_log_history = pd.DataFrame(trainer.state.log_history)

ruta_log_history = RUTA_RESULTADOS_CUADERNO_06 / "log_history_entrenamiento_limpio_v2.csv"

df_log_history.to_csv(
    ruta_log_history,
    index=False,
    encoding="utf-8-sig"
)

df_log_history

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_wer,eval_cer,eval_runtime,eval_samples_per_second,...,train_samples_per_second,train_steps_per_second,total_flos,train_loss,test_loss,test_wer,test_cer,test_runtime,test_samples_per_second,test_steps_per_second
0,0.000439,0.014323,1.200245e-07,0.024483,100,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.000147,0.021739,2.424985e-07,0.048966,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.000257,0.021970,3.649724e-07,0.073448,300,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.000134,0.014986,4.874464e-07,0.097931,400,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.000327,0.028028,6.099204e-07,0.122414,500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,0.000031,0.006170,1.262602e-08,7.956053,32500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
332,0.000034,0.007675,6.184172e-09,7.980536,32600,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
333,NaN,NaN,NaN,8.000000,32680,0.147089,0.451125,0.331113,332.2869,4.917,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
334,NaN,NaN,NaN,8.000000,32680,NaN,NaN,NaN,NaN,NaN,...,6.605,0.826,1.695337e+19,0.000489,NaN,NaN,NaN,NaN,NaN,NaN


## Comparación con modelo mejorado anterior

In [44]:
# Cargamos el modelo mejorado anterior para compararlo sobre el mismo test limpio v2
modelo_anterior = WhisperForConditionalGeneration.from_pretrained(
    str(RUTA_MODELO_MEJORADO_ANTERIOR)
).to(device)

modelo_anterior.generation_config.language = "spanish"
modelo_anterior.generation_config.task = "transcribe"

modelo_anterior.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="spanish",
    task="transcribe"
)

modelo_anterior.generation_config.suppress_tokens = []
modelo_anterior.config.use_cache = False

print("Modelo anterior cargado correctamente")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Modelo anterior cargado correctamente


In [45]:
# Creamos un trainer auxiliar para evaluar el modelo anterior
trainer_modelo_anterior = Seq2SeqTrainer(
    args=training_args,
    model=modelo_anterior,
    eval_dataset=dataset_v2_preparado["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [46]:
# Evaluamos el modelo mejorado anterior sobre el test limpio v2
metricas_modelo_anterior_test_v2 = trainer_modelo_anterior.evaluate(
    eval_dataset=dataset_v2_preparado["test"],
    metric_key_prefix="modelo_anterior_test_v2"
)

metricas_modelo_anterior_test_v2

Training Loss,Validation Loss,Epoch,Anterior Test V2 Loss,Anterior Test V2 Wer,Anterior Test V2 Cer
No log,No log,0,0.137131,0.478542,0.352143


{'modelo_anterior_test_v2_loss': 0.13713112473487854,
 'modelo_anterior_test_v2_wer': 0.47854175111471425,
 'modelo_anterior_test_v2_cer': 0.3521429013270904}

In [47]:
# Creamos una tabla comparativa entre ambos modelos
comparacion_modelos = pd.DataFrame([
    {
        "modelo": "Modelo mejorado anterior",
        "loss": metricas_modelo_anterior_test_v2["modelo_anterior_test_v2_loss"],
        "wer": metricas_modelo_anterior_test_v2["modelo_anterior_test_v2_wer"],
        "cer": metricas_modelo_anterior_test_v2["modelo_anterior_test_v2_cer"]
    },
    {
        "modelo": "Modelo reentrenado limpio v2",
        "loss": metricas_test_v2["test_loss"],
        "wer": metricas_test_v2["test_wer"],
        "cer": metricas_test_v2["test_cer"]
    }
])

comparacion_modelos

,modelo,loss,wer,cer
0,Modelo mejorado anterior,0.137131,0.478542,0.352143
1,Modelo reentrenado limpio v2,0.146049,0.480188,0.354950


In [48]:
# Calculamos la variación porcentual entre el modelo anterior y el nuevo
wer_anterior = comparacion_modelos.loc[
    comparacion_modelos["modelo"] == "Modelo mejorado anterior",
    "wer"
].values[0]

wer_nuevo = comparacion_modelos.loc[
    comparacion_modelos["modelo"] == "Modelo reentrenado limpio v2",
    "wer"
].values[0]

cer_anterior = comparacion_modelos.loc[
    comparacion_modelos["modelo"] == "Modelo mejorado anterior",
    "cer"
].values[0]

cer_nuevo = comparacion_modelos.loc[
    comparacion_modelos["modelo"] == "Modelo reentrenado limpio v2",
    "cer"
].values[0]

variacion_wer = ((wer_anterior - wer_nuevo) / wer_anterior) * 100
variacion_cer = ((cer_anterior - cer_nuevo) / cer_anterior) * 100

print("Variación WER (%):", round(variacion_wer, 2))
print("Variación CER (%):", round(variacion_cer, 2))

Variación WER (%): -0.34
Variación CER (%): -0.8


In [49]:
# Guardamos la comparación de modelos
ruta_comparacion_modelos = RUTA_RESULTADOS_CUADERNO_06 / "comparacion_modelos_limpio_v2.csv"

comparacion_modelos.to_csv(
    ruta_comparacion_modelos,
    index=False,
    encoding="utf-8-sig"
)

print("Comparación guardada en:", ruta_comparacion_modelos)
comparacion_modelos

Comparación guardada en: C:\Lara\resultados\06_reentrenamiento_limpio_v2\comparacion_modelos_limpio_v2.csv


,modelo,loss,wer,cer
0,Modelo mejorado anterior,0.137131,0.478542,0.352143
1,Modelo reentrenado limpio v2,0.146049,0.480188,0.354950


## Conclusiones

En este cuaderno se ha continuado el entrenamiento del modelo `whisper_base_mejorado` utilizando el dataset v2 con splits originales filtrados.

La limpieza realizada en el cuaderno 05 fue parcial. Principalmente se eliminaron algunas muestras problemáticas detectadas durante el análisis manual del conjunto de test. Por tanto, el conjunto de entrenamiento no cambia de forma significativa respecto al utilizado anteriormente.

Los resultados sobre el test filtrado muestran que el modelo reentrenado no mejora al modelo mejorado anterior:

- WER modelo mejorado anterior: 0.478542
- WER modelo reentrenado v2: 0.480188
- CER modelo mejorado anterior: 0.352143
- CER modelo reentrenado v2: 0.354950

La diferencia es pequeña, pero el nuevo entrenamiento produce un ligero empeoramiento.

Esto indica que seguir entrenando el modelo `whisper-base` sobre prácticamente el mismo conjunto de entrenamiento no aporta mejora con esta configuración. El modelo anterior sigue siendo la mejor versión obtenida hasta el momento.

El modelo reentrenado queda guardado como experimento, pero no sustituye al modelo `whisper_base_mejorado`.

Como siguiente paso, tendría más sentido probar una mejora de arquitectura, por ejemplo entrenar `openai/whisper-small` sobre el dataset disponible, o bien realizar una limpieza más amplia del conjunto de entrenamiento antes de volver a reentrenar.